# Generación de Texto con modelos GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion4/1-text-generation.ipynb)

En este notebook harémos uso de un modelo tipo GPT-2 pre-entrenado en idioma español que utilizaremos para generar texto a partir de un contexto inicial que proveerémos. Luego, harémos fine tuning a este modelo con un dataset de chistes en español y observar como cambia la generación de texto en función del dataset que utilicemos.

#### Referencias
- Dataset: https://huggingface.co/datasets/mrm8488/CHISTES_spanish_jokes
- [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246)
- [GPT2 Spanish](https://huggingface.co/DeepESP/gpt2-spanish)
- [Fine-Tune a non-Englush GPT-2 Model with Huggingface](https://www.philschmid.de/fine-tune-a-non-english-gpt-2-model-with-huggingface)

---
## 📖 Cómo leer este notebook (guía para no-programadores)

Este notebook es una **copia comentada** para que cualquiera pueda entenderlo,
incluso si nunca ha programado. La versión original, sin comentarios, está en la
carpeta `Taller_4/` junto a este archivo, y esta copia ejecuta **exactamente el mismo
código**, solo que con explicaciones extra.

### 📚 Mini-diccionario de términos

| Término | Traducción a lenguaje humano |
|---|---|
| **Modelo** | Un "cerebro digital" al que se le enseñó leyendo millones de textos. Aquí usamos GPT-2 ya entrenado en español. |
| **Token** | Un pedacito de texto: puede ser una palabra completa o un trozo de palabra. Los modelos no leen letras, leen tokens. |
| **Tokenizador** | El "traductor" que convierte texto humano ↔️ números que el modelo entiende. |
| **Tensor** | Una tabla de números (de 1, 2 o 3 dimensiones). Es el "idioma nativo" con el que piensa el modelo. |
| **Logits** | Los puntajes crudos que el modelo asigna a cada posible siguiente token. Número más alto = el modelo "le gusta más" ese token. |
| **Softmax** | Una fórmula que convierte los logits en **probabilidades** (números entre 0% y 100% que suman 100% en total). |
| **Generar texto** | Juego en bucle: predice la siguiente palabra → la agrega → predice la siguiente de nuevo → y así hasta terminar. |
| **GPU / CPU** | La GPU es un chip especial (el de los videojuegos) ideal para IA porque hace muchísimos cálculos en paralelo. |
| **Fine tuning** | Enviar al modelo a una "escuela especializada": le enseñamos con ejemplos nuevos (aquí, chistes) para ajustar su estilo. |
| **Dataset** | La colección de ejemplos con la que entrenamos. Aquí: miles de chistes en español. |
| **Época** | Una pasada completa del modelo por todo el dataset de entrenamiento. |

### 🗺️ Mapa del recorrido

1. Cargar el modelo GPT-2 en español y ver cómo predice la siguiente palabra.
2. Construir a mano una función que genere texto palabra por palabra.
3. Probar distintas estrategias de selección (greedy vs. muestreo aleatorio).
4. Afinar (fine tuning) el modelo con un dataset de chistes en español.
5. Comparar cómo cambia la generación antes y después del fine tuning.

💡 **Sugerencia:** lea primero cada celda de explicación (las que empiezan con 🧠),
luego mire el código. No necesita entender cada símbolo: con seguir la narrativa basta.


### 🧠 Explicación: Preparar las herramientas

Antes de cocinar hay que sacar los utensilios. Esta celda:
1. Le pide a Python que **no muestre avisos técnicos** (alertas que no son errores y solo ensucian la pantalla).
2. Detecta si estamos corriendo dentro de **Google Colab** (la plataforma gratuita de Google que se ejecuta en el navegador y presta computadoras con GPU).
3. Si estamos en Colab, quita un paquete llamado `torchvision` que no necesitamos y suele dar conflictos ahí.

Al final queda una variable `IN_COLAB` que vale `True` (sí estamos en Colab) o `False` (estamos en otra computadora). Más adelante la usaremos para ajustar el tamaño del entrenamiento.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PREPARACIÓN DEL ENTORNO (sacar la "caja de herramientas")
# ─────────────────────────────────────────────────────────────────

# 'warnings' es el paquete que muestra avisos técnicos (no son errores,
# solo alertas). Lo importamos solo para poder silenciarlo a continuación.
import warnings

# Con esta línea le decimos a Python: "no me muestres esos avisos",
# para que la salida quede limpia y veamos solo lo importante.
warnings.filterwarnings('ignore')

# ¿Estamos dentro de Google Colab? Lo averiguamos intentando importar su módulo.
try:
    import google.colab
    IN_COLAB = True
    # torchvision is problematic in Colab and we don't need it for this notebook, so we uninstall it to avoid issues.
    !pip uninstall -y torchvision
except ImportError:
    # Si la importación falla, es que NO estamos en Colab (por ejemplo,
    # en la computadora local). Lo anotamos y seguimos sin problema.
    IN_COLAB = False


## Generative pre-training Transformer - GPT

![](https://github.com/Ohtar10/icesi-nlp/blob/main/assets/gpt.png?raw=1)

Los modelos tipo GPT, introducidos por Radfor, et.al., de OpenAI, al igual que los modelos BERT, hacen uso extensivo de la arquitectura de transformers como hemos estado viendo. Las diferencias claves se podrían resumir en:

1. GPT utiliza bloques de **Transformer Decoder** encadenados, mientras que el modelo BERT utiliza bloques de *Transformer Encoder*
2. GPT se centra en la generación de texto basado en un contexto, la tarea principal es la predicción del siguiente token en la secuencia, mientras que BERT se centra en el completado de partes de una secuencia, en función de un contexto anterior y posterior a la secuencia de entrada. Entonces BERT se centra en la construicción de representación de lenguage, mientras que GPT se centra en la generación de texto en función de un contexto.

Sin embargo, ambos se basan en la misma premisa de pre-entrenar el modelo en tareas no-supervisadas o semi-supervisadas para que el modelo aprenda las representaciones semánticas del lenguage y luego al modelo se le pueda hacer fine tuning a tareas posteriores.

### 🧠 Explicación: Cargar el modelo y su traductor

Aquí descargamos y ponemos en marcha dos piezas:

- **El tokenizador**: piense en él como un traductor. Toma texto humano ("Había una vez") y lo convierte en números, porque el modelo no entiende letras, entiende números. También hace el trabajo inverso (números → texto).
- **El modelo**: `mrm8488/spanish-gpt2` es un GPT-2 que **ya viene entrenado en español** por alguien más. No lo creamos nosotros: lo descargamos listo para usar, como comprar un automóvil ya armado.

Dos detalles técnicos que parecen crípticos pero son simples:
- `device = "cuda" if ... else "cpu"` → pregunta: ¿hay GPU disponible? Si sí, la usamos (mucho más rápido); si no, usamos el procesador normal.
- Las líneas del `pad_token` → algunos modelos necesitan un "carácter de relleno" para igualar frases de distinta longitud, y este modelo no trae uno definido; le asignamos el "fin de texto" (`eos_token`) para que no falle.

In [2]:
import torch  # PyTorch: el 'motor' que hace los cálculos matemáticos de la IA.
import transformers  # Hugging Face Transformers: librería que trae modelos ya entrenados listos para usar.
from transformers import AutoModelForCausalLM, AutoTokenizer


# ¿GPU o CPU? La GPU es un chip especial, el mismo tipo que usan los videojuegos,
# excelente para IA porque hace muchísimos cálculos a la vez.
# Si hay una disponible usamos 'cuda' (su nombre técnico); si no, el procesador normal.
device = "cuda" if torch.cuda.is_available() else "cpu"

# El 'nombre del modelo' es la dirección donde vive en internet (Hugging Face).
model_name = "mrm8488/spanish-gpt2"

# El TOKENIZADOR: el traductor texto-humano → números. Lo descargamos.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# El MODELO en sí: el GPT-2 en español ya entrenado. Lo descargamos
# y con .to(device) lo colocamos en la GPU (si existe) o en la CPU.
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# A veces el tokenizador no trae definido un 'token de relleno' (pad),
# que se usa para igualar la longitud de frases distintas al procesarlas juntas.
if tokenizer.pad_token is None:
  # Solución sencilla: usar el 'token de fin de texto' como relleno.
  tokenizer.pad_token = tokenizer.eos_token

# Le informamos al modelo cuál es el número que representa ese token de relleno.
model.config.pad_token_id = tokenizer.pad_token_id

# Ajustamos el tamaño de la 'tabla de vocabulario' del modelo para que
# coincida exactamente con el tamaño del vocabulario del tokenizador.
model.resize_token_embeddings(len(tokenizer))

# Al escribir solo 'model' como última línea, el notebook muestra
# la ficha técnica completa del modelo: cuántas capas tiene, etc.
model


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2663.26it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: mrm8488/spanish-gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50266, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50266, bias=False)
)

### 🧠 Explicación: Mirar bajo el capó del modelo

Esta celda lista los **módulos internos** del modelo: cada uno es un bloque del "cerebro" (capas de atención, redes neuronales, etc.).

No hay que memorizarlos; la idea es ver que GPT-2 no es una caja negra única, sino **una torre de bloques apilados**: cada bloque procesa el texto un poco más profundo que el anterior, y por eso entiende mejor el contexto.

In [3]:
# Recorremos todos los 'módulos' (componentes internos) del modelo
# y guardamos sus nombres en una lista. 'named_modules()' entrega
# pares (nombre, componente); aquí solo nos interesa el nombre '_'.
modules = [m for m, _ in model.named_modules()]
modules  # Mostrar la lista: cada línea es un bloque interno del 'cerebro'.


['',
 'transformer',
 'transformer.wte',
 'transformer.wpe',
 'transformer.drop',
 'transformer.h',
 'transformer.h.0',
 'transformer.h.0.ln_1',
 'transformer.h.0.attn',
 'transformer.h.0.attn.c_attn',
 'transformer.h.0.attn.c_proj',
 'transformer.h.0.attn.attn_dropout',
 'transformer.h.0.attn.resid_dropout',
 'transformer.h.0.ln_2',
 'transformer.h.0.mlp',
 'transformer.h.0.mlp.c_fc',
 'transformer.h.0.mlp.c_proj',
 'transformer.h.0.mlp.act',
 'transformer.h.0.mlp.dropout',
 'transformer.h.1',
 'transformer.h.1.ln_1',
 'transformer.h.1.attn',
 'transformer.h.1.attn.c_attn',
 'transformer.h.1.attn.c_proj',
 'transformer.h.1.attn.attn_dropout',
 'transformer.h.1.attn.resid_dropout',
 'transformer.h.1.ln_2',
 'transformer.h.1.mlp',
 'transformer.h.1.mlp.c_fc',
 'transformer.h.1.mlp.c_proj',
 'transformer.h.1.mlp.act',
 'transformer.h.1.mlp.dropout',
 'transformer.h.2',
 'transformer.h.2.ln_1',
 'transformer.h.2.attn',
 'transformer.h.2.attn.c_attn',
 'transformer.h.2.attn.c_proj',
 'tran

Observermos un ejemplo de generación simple.

### 🧠 Explicación: El modelo predice la siguiente palabra — un solo paso

Este es **el corazón conceptual de todo el notebook**. Le damos al modelo la frase *"Había una vez"* y le preguntamos: **"¿qué palabra viene después?"**

Lo que hace el código, paso a paso:
1. Convierte la frase en tokens (números) con el tokenizador.
2. El modelo procesa esos números y devuelve **logits**: un puntaje para CADA palabra de su vocabulario (son ~50.000 palabras, ¡y se puntúan todas!).
3. Con `softmax` convierte esos puntajes en probabilidades que suman 100%.
4. Ordena las palabras de mayor a menor probabilidad y muestra las **10 más probables** con su porcentaje.

Verá una impresión del tipo `{'una': '3.5%', 'de': '3.1%'...}`. Cada entrada es una palabra candidata para continuar la historia. **Este mismo cálculo es lo que hace ChatGPT en su cabeza millones de veces**: predecir, elegir, repetir.

In [4]:
# La frase inicial que le daremos al modelo (el 'contexto' de la historia).
text = "Había una vez"

# ¿Cuántas palabras candidatas más probables queremos ver? Elegimos 10.
best = 10

# 'torch.no_grad()' le dice al modelo: "no estás aprendiendo nada, solo
# respondes". Esto ahorra memoria y es la manera correcta de solo preguntar.
with torch.no_grad():
    # 1) Traducir la frase a tokens (números). 'return_tensors=pt' dice
    #    'devuélvela en formato PyTorch'. .to(device) la manda a la GPU/CPU.
    tokens = tokenizer(text, return_tensors='pt')['input_ids'].to(device)
    print("Dimensiones de la entrada:", tokens.shape)

    # 2) Pasar la frase por el modelo. Nos devuelve 'logits': puntajes para
    #    cada palabra posible, no solo para una, sino en TODAS las posiciones.
    output = model(input_ids=tokens)
    print("Dimensiones de la salida:", output.logits.shape)

    # 3) A nosotros solo nos interesa la ÚLTIMA posición (la que predice
    #    qué viene DESPUÉS de 'Había una vez'). [0, -1, :] significa:
    #    'del primer (0) texto, la última (-1) posición, y todo (:) lo demás'.
    output = output.logits[0, -1, :]
    print("Dimensiones del último token de la secuencia:", output.shape)

    # 4) Convertir los puntajes en probabilidades que suman 100%.
    probs = torch.softmax(output, dim=-1)
    print("Dimensiones de la probabilidad de los tokens:", probs.shape)

    # 5) Ordenar las palabras por probabilidad, de la más alta a la más baja.
    sorted_probs = torch.argsort(probs, dim=-1, descending=True)

    # 6) Mostrar las 10 más probables: 'palabra: porcentaje%'.
    print({tokenizer.decode(token): f"{prob.cpu().numpy() * 100:.2f}%" for token, prob in zip(sorted_probs[:best], probs[sorted_probs[:best]])})


Dimensiones de la entrada: torch.Size([1, 3])
Dimensiones de la salida: torch.Size([1, 3, 50266])
Dimensiones del último token de la secuencia: torch.Size([50266])
Dimensiones de la probabilidad de los tokens: torch.Size([50266])
{' un': '30.17%', ' una': '25.06%', ',': '14.50%', ' en': '8.18%', ' que': '5.96%', '...': '5.87%', ' dos': '1.74%', '.': '0.72%', ' tres': '0.67%', ' uno': '0.34%'}


## Implementando una función de generación

Ahora, la idea es que este modelo nos sirva para generar texto de forma recurrente e incremental. En la última capa de los modelos tipo GPT encontrarémos un tensor con forma $(b, s, v)$, donde:

- $b$: Es el tamaño del bache, o la cantidad de secuencias a procesar.
- $s$: Es la longitud de la secuencia de entrada.
- $v$: Es el tamaño del vocabulario del modelo, cuantos tokens soporta.

Pero este es el tensor de salida, por qué tiene la forma de la secuencia de entrada?, porque cada posición en la salida corresponde a la la predicción del siguiente token de esa posición en la secuencia de entrada. En otras palabras, lo que obtenemos como predicción, es una secuencia de igual tamaño a la de entrada, movida un token hacia adelante, lo que efectivamente nos predice un solo token a la vez y por ende, el token que nos insteresa, es el último.

Lo que obtenemos en este tensor es además los logits de TODO el vocabulario del modelo, con los cuales podemos calcular las probabilidades de que cada uno sea el que continue en la secuencia. Hay varias formas de decodificar el siguiente token, la más fácil de implementar sería una decodificación codiciosa (greedy) del siguiente token, que consiste simplemente en seleccionar el token con la probabilidad más alta. Este es un enfoque simple y efectivo para algunos casos, pero al mismo tiempo sufre de poca variabilidad e incluso puede caer en generación repetitiva.

Otra opción es el muestreo, ya que justamente podemos obtener probabilidades del siguiente token, lo más lógico sería muestrear con esas opciones de probabilidad, de este modo podemos obtener mayor diversidad a la hora de generar el texto, al costo eso si de que haya mayor aleatoridad ya que se le daría la oportunidad a incluso tokens con baja probabilidad, de ser seleccionados.

Otra opción podría ser un balanceo entre una decodificación greedy y una por muestreo, en función de otro hiperparámetro que podemos definir. Esta sería una técnica muy común en el contexto de Reinforcement Learning llamade e-greedy. Se hace la aclaración de que en este ejemplo no harémos nada de RL, solamente se hace mención de esta técnica para balancear entre explotación y exploración.

### 🧠 Explicación: Construir nuestra propia máquina de escribir (la función `generate`)

En la celda anterior el modelo predijo **una sola** palabra. Para escribir un párrafo completo hay que repetir el truco: predecir → elegir → agregar la palabra al final → volver a predecir… igual que quien juega a completar frases en cadena.

Antes de leer el código, recuerde el dilema clave — **¿cómo elegimos la siguiente palabra?**

| Estrategia | ¿Cómo elige? | Consecuencia |
|---|---|---|
| **Codiciosa (greedy)** | Siempre toma la palabra con MAYOR probabilidad | Precisa pero repetitiva/aburrida |
| **Muestreo (sampling)** | Sortea la palabra según las probabilidades (cualquiera puede salir, incluso una rara) | Creativa pero a veces incoherente |
| **ε-greedy (la de esta función)** | Mezcla las dos anteriores: con probabilidad `eps` va a lo seguro, con `1-eps` sortea | Balance configurable |

El parámetro `eps` es entonces la "perilla" que controla el temperamento del modelo:
- `eps=1.0` → 100% codicioso: siempre el mejor token, **siempre el mismo resultado**.
- `eps=0.5` → la mitad de las veces va a lo seguro, la otra mitad sortea.
- `eps=0.0` → 100% sorteo puro.

💡 El parámetro `return_iterations=True` pide a la función que además devuelva una tabla con cada paso (qué opciones evaluó y en qué momento) — útil para *ver* la decisión interna. No todo notebook necesita esto: aquí se hace con fines didácticos.

In [5]:
import torch.nn as nn  # Piezas para construir redes neuronales (solo la usamos para el tipo del parámetro).
import numpy as np  # Herramientas matemáticas (la usamos para sortear un número aleatorio).
import pandas as pd  # Tablas tipo Excel (la usamos para mostrar el historial de iteraciones).
from typing import Optional, Tuple  # Solo para 'anotar' los tipos de datos de la función.
from transformers.tokenization_utils_base import PreTrainedTokenizerBase  # Tipo del tokenizador, solo para anotación.


def generate(
        model: nn.Module,
        tokenizer: PreTrainedTokenizerBase,
        start: str,          # La frase inicial con la que arranca la historia.
        max_length: int = 100,   # ¿Cuántas palabras (tokens) máximo agregamos?
        eps: float = 0.5,    # La 'perilla' del balance codicioso/sorteo (ver explicación arriba).
        top_n: int = 5,      # Si guardamos el historial, ¿cuántas opciones guardamos por paso?
        return_iterations: bool = False,  # ¿Devolver también la tabla con 'el pensamiento' paso a paso?
        device: str = "cpu") -> Tuple[str, Optional[pd.DataFrame]]:

    # La lista 'output' acumula el texto resultado. Empieza con la frase inicial.
    output = [start]
    iterations = []  # Aquí guardaremos el historial de pasos (si lo pedimos).
    with torch.no_grad():  # 'Solo responder, no aprender' (ahorra memoria).
        # Traducir la frase inicial a números.
        input_ids = tokenizer(output[-1], return_tensors='pt')['input_ids'].to(device)
        # EL BUCLE: repetir este proceso max_length veces (un token por vuelta).
        for _ in range(max_length):
            # Tomamos los logits producidos por la última capa del modelo
            # Estos corresponden al siguiente token por cada posición de la cadena
            logits = model(input_ids=input_ids).logits
            # Por lo tanto, el que nos interesa es el último, que correspondería a la
            # predicción del siguiente token después del final de la cadena original
            # A este aplicamos un softmax para obtener las probabilidades por cada
            # token del vocabulario para estar presente en la cadena.
            probs = torch.softmax(logits[0, -1, :], dim=-1)
            # Simplemente ordenamos por probabilidad de forma descendente
            sorted_tokens = torch.argsort(probs, dim=-1, descending=True)

            # Utilizamos una política tipo e-greedy para obtener el siguiente token de la secuencia
            # Un eps>=1 quiere decir que siempre se va seleccionar el token de forma 'greedy', es decir
            # siempre se toma el token con probabilidad más alta.

            # Un eps=0 quiere decir que siempre se va a muestrear el siguiente token en función
            # de las probabilidades de cada token

            # Un 0<eps<1 va a balancear de forma binomial entre tomar el token con la
            # probabilidad más alta y muestrear el token en función de sus probabilidades.

            # Sorteamos un número aleatorio entre 0 y 1. Si sale MENOR que eps → modo codicioso.
            if np.random.random_sample(1)[0] < eps:
                # Se toma el mejor token (el de mayor probabilidad).
                next_token = sorted_tokens[0].unsqueeze(dim=0)
            else:
                # Se muestrea el token de la probabilidad de distribución:
                # 'multinomial' es como girar una ruleta donde cada token
                # ocupa un espacio proporcional a su probabilidad.
                next_token = torch.multinomial(probs, 1)

            if return_iterations:
                # Mantenemos pista de todas las iteraciones para análisis
                iteration = {'input': ''.join(output)}
                best_n = sorted_tokens[:top_n].cpu().tolist()
                choices = {f'Choice #{choice+1}': f'{tokenizer.decode(token)} ({prob:.4f})' for choice, (token, prob) in enumerate(zip(best_n, probs[best_n].cpu().tolist()))}
                iteration.update(choices)
                iterations.append(iteration)

            # Agregamos la palabra elegida (convertida de número a texto) a la historia.
            output.append(tokenizer.decode(next_token))
            # Y pegamos su versión numérica al final de los números de entrada,
            # de modo que en la próxima vuelta del bucle el modelo la 'vea'
            # y prediga la palabra que viene después de ella.
            input_ids = torch.cat([input_ids, next_token.unsqueeze(dim=0)], dim=-1)

        # Terminó el bucle: unimos todas las piezas de texto en una sola cadena.
        output_text = ''.join(output)
        if not return_iterations:
            return output_text, None
        else:
            df = pd.DataFrame(iterations)
            return output_text, df


Ahora observemos que pasa cuando generamos texto con nuestra función y algunos parámetros.

Primero, observemos que pasa cuando pasamos un `eps=1` que quiere decir que la generación va a ser de tipo greedy:

### 🧠 Explicación: Primera prueba — generación 100% codiciosa (`eps=1.0`)

Invocamos nuestra función con `eps=1.0`, o sea **siempre** elige la palabra más probable, sin dejar lugar al azar.

Consecuencia interesante: aunque corra esta celda 100 veces, **siempre obtiene exactamente el mismo texto**. Siempre toma el camino "más seguro" en cada cruce, así que el camino entero es determinista.

La tabla debajo (`iterations_df`) es la bitácora de la conversación: cada fila es un paso del bucle, muestra cuál era el texto acumulado hasta ese punto (`input`) y las 10 opciones que el modelo consideraba (`Choice #1` es la más probable, que es la que se elige). **Mire esa columna para ver cómo la frase crece palabra a palabra.**

In [6]:
# Generamos con:
#   eps=1.0            → 100% codicioso (siempre la palabra más probable)
#   max_length=15      → máximo 15 tokens nuevos
#   top_n=10           → guardar 10 opciones por paso en la bitácora
#   return_iterations  → sí, queremos ver 'el pensamiento' paso a paso
output_text, iterations_df = generate(model, tokenizer, text, max_length=15, eps=1.0, top_n=10, return_iterations=True, device=device)

# Mostrar el texto generado.
print(output_text)

# Mostrar la bitácora: cada fila = un paso del bucle con las opciones evaluadas.
iterations_df.head(15)


Había una vez un hombre que se llamaba a sí mismo "El hombre de la bolsa".


,input,Choice #1,Choice #2,Choice #3,Choice #4,Choice #5,Choice #6,Choice #7,Choice #8,Choice #9,Choice #10
0,Había una vez,un (0.3017),una (0.2506),", (0.1450)",en (0.0818),que (0.0596),... (0.0587),dos (0.0174),. (0.0072),tres (0.0067),uno (0.0034)
1,Había una vez un,hombre (0.1397),rey (0.0564),niño (0.0535),joven (0.0460),chico (0.0347),pequeño (0.0321),príncipe (0.0192),tipo (0.0166),muchacho (0.0164),viejo (0.0148)
2,Había una vez un hombre,que (0.3329),llamado (0.1111),", (0.0773)",y (0.0529),... (0.0431),de (0.0326),en (0.0320),con (0.0285),muy (0.0203),. (0.0126)
3,Había una vez un hombre que,se (0.1276),tenía (0.0634),vivía (0.0624),no (0.0623),era (0.0422),estaba (0.0276),amaba (0.0263),", (0.0255)",fue (0.0238),había (0.0182)
4,Había una vez un hombre que se,llamaba (0.1948),había (0.0505),enamoró (0.0370),sentó (0.0296),convirtió (0.0198),casó (0.0181),sentía (0.0149),hizo (0.0149),fue (0.0146),encontró (0.0121)
5,Había una vez un hombre que se llamaba,a (0.0438),... (0.0425),John (0.0128),""" (0.0104)",José (0.0101),David (0.0097),Peter (0.0088),Juan (0.0086),Henry (0.0081),A (0.0074)
6,Había una vez un hombre que se llamaba a,sí (0.8622),si (0.1250),él (0.0034),uno (0.0008),Sí (0.0007),un (0.0006),su (0.0005),mismo (0.0005),la (0.0004),engaño (0.0004)
7,Había una vez un hombre que se llamaba a sí,mismo (0.9944),propio (0.0017),misma (0.0015),m (0.0003),mismo (0.0002),a (0.0002),solo (0.0002),", (0.0002)",sí (0.0001),y (0.0001)
8,Había una vez un hombre que se llamaba a sí mismo,""" (0.1287)",el (0.0575),... (0.0179),un (0.0161),señor (0.0145),Señor (0.0144),Dios (0.0132),rey (0.0110),", (0.0099)",Capitán (0.0095)
9,Había una vez un hombre que se llamaba a sí mi...,El (0.0862),el (0.0581),Padre (0.0191),Dios (0.0169),Señor (0.0150),La (0.0131),señor (0.0124),Rey (0.0114),S (0.0096),Sr (0.0083)


Observamos como el input progresa a la vez que las opciones de tokens que hay. Sin importar cuantas veces invoquemos a la función con los mismos parámetros, siempre vamos a obtener los mismos resultados.

Ahora, observemos que pasa si introducimos exploración al reducir el `eps=0.5`, lo cual nos dice que aproximadamente la mitad de las veces va a elegir el siguiente token muestreando y la otra mitad explotando.

### 🧠 Explicación: Segunda prueba — dejamos entrar el azar (`eps=0.5`)

Ahora bajamos la perilla a `eps=0.5`: la mitad de las veces el modelo va a lo seguro y la otra mitad **sortea** la siguiente palabra con una ruleta ponderada por probabilidades.

Ejecute esta celda varias veces seguidas: **cada corrida dará un texto distinto** (a veces coherente, a veces raro). Eso es la exploración: compramos variedad al costo de controlar la calidad. Este es exactamente el dilema que menciona la celda siguiente y que también vive dentro de ChatGPT (aunque con técnicos más refinados).

In [7]:
# Ahora con eps=0.5: la mitad de las veces codicioso, la otra mitad sortea.
# Ejecute esta celda varias veces: cada resultado será diferente.
# (El '_, ' a la izquierda significa 'no me interesa guardar la bitácora ahora')
output_text, _ = generate(model, tokenizer, text, max_length=100, eps=0.5, device=device)

print(output_text)


Había una vez una mujer que se llamaba la Reina de la Jungla.Ella era una mujer muy especial a quien le gustaba patear a sus cerdos.Y ella le cobraba cien al mes por eso.Y su cerdo era muy especial.El rey de la Jungla tenía un cerdo.Y el cerdo era tan especial, que él quería el cerdo.Y el rey de la Jungla estaba tan agradecido por el cerdo.Pero el cerdo era muy frío.Así que él le dio el cerdo.Y el cerdo


En este caso, cada vez que invoquemos a la función, vamos a obtener una respuesta diferente, a veces más coherente y otras veces menos. Vale la pena realizar varias pruebas para observar los resultados hasta encontrar un balance.

### Generando texto con las utilidades del modelo

Ahora, la clase de Huggingface implementa la función `generate` que hace la labor de generación por nosotros, incluyendo las opciones de muestreo y explotación como hemos observado. Solo que además permite otra serie de parámetros y opciones para controlar la generación de texto. Para más información se recomienda estudiar:

- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246), Capitulo 5
- https://huggingface.co/docs/transformers/v4.41.3/en/main_classes/text_generation#transformers.GenerationConfig
- https://huggingface.co/docs/transformers/v4.41.3/en/main_classes/text_generation#transformers.GenerationMixin.generate

### 🧠 Explicación: Usando el generador profesional de Hugging Face

Hasta aquí construimos la función `generate` **a mano** para entender cómo funciona por dentro. Pero la librería Hugging Face ya trae su propia función `generate` integrada al modelo, con más "perillas" profesionales:

- `do_sample=True` → activa el sorteo aleatorio (como nuestro muestreo).
- `temperature=0.5` → controla lo "atrevido" del sorteo: temperatura baja (0.2) hace el modelo conservador; temperatura alta (1.5) lo hace creativo/desatado. Es otra forma de lo mismo que hicimos con `eps`.
- `top_k=0` → considera todas las palabras del vocabulario en el sorteo (otros valores recortan la ruleta a las K más probables).

💡 Nota: la frase de entrada `tokens` ya la generamos en la celda 6; aquí reutilizamos esa traducción a números en vez de recomputarla.

In [8]:
# La función oficial 'generate' del modelo hace el mismo trabajo que la nuestra,
# pero con parámetros más finos:
#   pad_token_id=...  → cómo rellenar (toca decirlo, recuerde que este modelo no lo traía definido)
#   max_length=100    → hasta 100 tokens en total
#   do_sample=True    → activar el sorteo aleatorio
#   temperature=0.5   → qué tan atrevido es el sorteo (bajo = conservador, alto = creativo)
#   top_k=0           → no recortar la ruleta: todas las palabras del vocabulario participan
output = model.generate(tokens, pad_token_id=tokenizer.eos_token_id, max_length=100, do_sample=True, temperature=0.5, top_k=0)

# La salida llega en números; hay que traducirla de vuelta a texto.
print(tokenizer.decode(output[0]))


Había una vez un hombre que vivía en una casa de la que nadie conocía.Había un hombre que vivía en una casa en la que nadie conocía.Había un hombre que vivía en una casa en la que nadie conocía.Había un hombre que vivía en una casa en la que nadie conocía.Había un hombre que vivía en una casa en la que nadie conocía.Había un hombre que vivía en una casa en la que nadie conocía.Había un hombre que vivía en una casa en la que nadie conocía.


## Fine tuning

Ahora, intentemos hacer fine tuning a nuestro modelo. Intentemos entrenarlo en un corpus de chistes en idioma español y ver como la narrativa de su output cambia.

##### Nota
Lastimosamente, este dataset es muy pequeño y la distribución del texto es muy diferente a la distribución de texto con la cual fue entrenado el modelo original, por lo que no se esperan resultados significativamente ejemplares. Sin embargo, el objetivo es observar como cambia la generación del texto una vez lo entrenamos en un conjunto especializado.

### 🧠 Explicación: El plan del fine tuning (mandar al modelo a la escuela de chistes)

Hasta ahora usamos el modelo "tal cual vino de fábrica". Ahora haremos **fine tuning**: entrenarlo un poco más con un dataset específico, en este caso **miles de chistes en español**.

La analogía: el modelo ya estudió una carrera general de español (con millones de libros y páginas web). Ahora le haremos un diplomado en chistes: le mostraremos los chistes muchas veces para que imite ese estilo.

⚠️ Nota honesta que también dice la celda original: el dataset es pequeño y muy distinto al texto general con que se entrenó, así que **no esperamos chistes brillantes** — el objetivo académico es *observar el cambio de estilo*, no fabricar un comediante.

Esta celda descarga el dataset (`load_dataset`).

In [9]:
from datasets import load_dataset  # Utilidad oficial de Hugging Face para descargar datasets listos.

# Descargamos el dataset: miles de chistes en español, ya publicados en Hugging Face.
dataset = load_dataset("mrm8488/CHISTES_spanish_jokes")

# Mostrar el resumen: cuántos ejemplos hay, sus columnas, etc.
dataset


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'keywords', 'funny', 'category'],
        num_rows: 2419
    })
})

### 🧠 Explicación: Espiar el primer chiste

Antes de usar un dataset siempre se **espea**: le echamos un vistazo a un ejemplo para ver cómo vienen los datos (qué columnas hay, qué forma tiene un chiste).

In [10]:
# Mostrar el primer chiste del conjunto de entrenamiento.
dataset['train'][0]


{'id': 0,
 'text': '- ¡Rápido, necesitamos sangre!\n- Yo soy 0 positivo.\n- Pues muy mal, necesitamos una mentalidad optimista.',
 'keywords': 'sangre',
 'funny': 1,
 'category': 'otros'}

### 🧠 Explicación: Ver los datos como una tabla de Excel

Aquí cambiamos el dataset a formato `pandas` (el "Excel" de Python) para verlo como tabla: cada fila es un chiste, cada columna un dato. Ver 10 filas ayuda a notar patrones (¿son cortos? ¿hay duplicados? ¿qué estilo tienen?) — paso que **siempre** conviene hacer antes de entrenar nada.

In [11]:
# Cambiar el dataset al formato 'pandas' (tabla tipo Excel).
dataset.set_format('pandas')

# Convertir el conjunto de entrenamiento a tabla y ver las primeras 10 filas.
df = dataset['train'].to_pandas()
df.head(10)


,id,text,keywords,funny,category
0,0,"- ¡Rápido, necesitamos sangre!\n- Yo soy 0 pos...",sangre,1,otros
1,1,- ¿Cuál es el mejor portero del mundial? \n- E...,"futbol,porteros",1,otros
2,2,El otro día unas chicas llamarón a mi puerta y...,"dinero,agua",1,otros
3,3,"- Andresito, ¿qué planeta va después de Marte?...",planetas,1,profesiones
4,4,- ¿Por qué Bob Esponja no va al gimnasio? \n- ...,"esponja,gimnasios",1,otros
5,5,Van dos ciegos y le dice uno al otro: \n- Ojal...,ciegos,1,otros
6,6,Noticia de última hora!! \n\nMuere una suegra ...,"canarias,coches,noticias",2,familia
7,7,"– Mamá, mamá, en el colegio dicen que estoy lo...","locos,sillas",1,familia
8,8,"– Mamá, mamá, ¿me haces un bocata de jamón?\n–...","madres,jamón",1,otros
9,9,- Qué pasa si te expulsan de cuatro univerdade...,"universitarios,universidades",1,otros


### 🧠 Explicación: ¿De qué tamaño son los chistes?

Pregunta legítima: para entrenar, ¿los textos son largos o cortos? Esta celda cuenta las **palabras por chiste** (divide el texto en palabras y cuenta cuántas salieron) y calcula la **mediana** (el valor del medio: la mitad de los chistes tienen menos palabras, la otra mitad más).

Saldrá **31**: es decir, los chistes son textos cortos. Esto confirma que el estilo de este dataset es muy distinto al de los libros/web con que se entrenó el modelo — por eso, tras el fine tuning, veremos un cambio de estilo marcado.

In [12]:
# Crear una columna nueva: 'Palabras por chiste'.
# .str.split() divide el texto en palabras; .apply(len) cuenta cuántas quedaron.
df['Palabras por chiste'] = df['text'].str.split().apply(len)

# Calcular la mediana: el punto donde el 50% de los chistes están por debajo.
df['Palabras por chiste'].median()


np.float64(31.0)

Aquí podemos observar que la mediana de longitud en terminos de palabras es de 31. Esto es esperado, pues los chistes deben ser cortos por naturaleza. Por otra parte, es bastante claro que el corpus original del modelo pre-entrenado contenía texto muy diferente a este, por lo que la calidad de los resultados, sin hacer mayores modificaciones puede que no sea buena.

Sin embargo, a manera demostrativa, continuarémos con el ejercicio, prepararémos el conjunto de datos para entrenamiento.

### 🧠 Explicación: La receta de preparación de datos (todavía no se cocina)

Esta celda **define una función, pero no la ejecuta todavía** (por eso no hay resultado). Es lo que en programación se llama una *función-fábrica*: `preprocess_function(64)` devuelve otra función especializada que sabe procesar ejemplos con máximo 64 tokens.

¿Qué hace esa función interna? Toma cada chiste, lo traduce a números (tokens) y **lo recorta si pasa de 64 tokens** (`truncation=True`). ¿Por qué 64? Recuerde la mediana: 31 palabras ≈ 40-50 tokens, así que 64 cubre casi todos los chistes completos sin desperdiciar memoria con textos gigantes.

In [13]:
# Función-fábrica: la llamamos con un máximo de longitud (max_len),
# y ella nos 'fabrica' una función procesadora personalizada con ese límite.
def preprocess_function(max_len):
    def _preprocess_function(examples):
        # Traducir cada chiste (texto) a números:
        #   max_length=max_len → si pasa del límite, recortarlo
        #   truncation=True    → activar ese recorte automático
        return tokenizer(examples['text'], max_length=max_len, truncation=True)
    return _preprocess_function


Los modelos GPT no esperan otra cosa más que los `input_ids`, por lo que retirarémos todas las demás columnas del dataset ya que no nos son de utilidad en este momento.

### 🧠 Explicación: Ejecutar la preparación (aquí sí se cocina)

Esta celda aplica la receta anterior a TODOS los chistes, en 4 pasos:

1. **`reset_format()`**: volvemos al formato normal del dataset (estábamos en modo Excel).
2. **`.map(...)`**: con la función recién creada, traduce los miles de chistes a números, con límite de 64 tokens.
3. **`remove_columns(...)`**: deja solo las columnas que el modelo necesita (`input_ids` = los números, `attention_mask` = cuáles posiciones son texto real vs. relleno). GPT para generar texto no pide nada más.
4. **`train_test_split(train_size=0.9)`**: reparte el dataset en **90% para entrenar** y **10% para examen**. ¿Por qué separar? Por la misma razón que en el colegio lo exámenes se hacen con preguntas nuevas: si el modelo estudiara los mismos textos con los que se le evalúa, la nota sería tramposa y no sabríamos si aprendió de verdad.

Al final, `set_format('torch')` pone las tablas en el formato que el motor de entrenamiento espera.

In [14]:
# 1) Volver del formato 'pandas' al formato normal del dataset.
dataset.reset_format()

# 2) Aplicar la receta a TODOS los chistes: traducirlos a números,
#    con límite de 64 tokens. batched=True procesa varios a la vez (más rápido).
tokenized_dataset = dataset['train'].map(preprocess_function(max_len=64), batched=True)

# 3) Quedarnos SOLO con las columnas útiles para GPT:
#    'input_ids' (los números) y 'attention_mask' (cuál posición es texto real
#    vs. relleno). Todo lo demás se descarta.
tokenized_dataset = tokenized_dataset.remove_columns([col for col in tokenized_dataset.column_names if col not in ['input_ids', 'attention_mask']])

# 4) Repartir: 90% para estudiar (train), 10% para el examen (test).
tokenized_dataset = tokenized_dataset.train_test_split(train_size=0.9)

# 5) Convertir todo a tensores de PyTorch (el formato que el entrenador espera).
tokenized_dataset.set_format('torch')

# Mostrar el resumen final: tamaños de train/test y columnas.
tokenized_dataset


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2177
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 242
    })
})

Finalmente procedemos a definir el entrenamiento. Observaremos que es muy similar a como entrenamos a BERT.

### 🧠 Explicación: Los ajustes del entrenamiento y el "entrenador"

Igual que al hornear hay que fijar temperatura y tiempo del horno, para entrenar hay que fijar los **hiperparámetros**. Los principales:

| Parámetro | Significado humano |
|---|---|
| `num_train_epochs=10` | El modelo leerá el dataset completo **10 veces** (10 pasadas = 10 épocas). |
| `learning_rate=2e-5` | Qué tanto corrige sus "creencias" en cada paso. Es chico a propósito: al modelo ya está educado y solo le damos retoques (un ajuste brusco "borraría" lo que ya sabe). |
| `batch_size` | Cuántos chistes procesa de un golpe. En Colab (`IN_COLAB`) hay GPU potente → 32 a la vez; en una computadora normal → solo 2. |
| `weight_decay=0.01` | Un freno anti-exceso: evita que el modelo se "aprenda de memoria" los chistes en vez de aprender el estilo. |
| `eval_strategy='epoch'` | Al final de cada época, hace un examen con el 10% apartado y anota la nota (la *pérdida* / loss). |
| `load_best_model_at_end=True` | Si el modelo empeora en épocas tardías, conserva la mejor versión. |

El **`DataCollatorForLanguageModeling`** con `mlm=False` es el director de la escuela y define el juego de estudio: *le das al modelo el chiste con la última palabra tapada y le pides adivinarla*. (`mlm=False` = "no es estilo BERT con palabras tapadas al azar; es estilo GPT, predecir el siguiente token").

💡 En Taller 3 hicimos algo análogo con BERT (+ clasificadores encima). Aquí la mecánica de `Trainer` es la misma; cambia la tarea: ya no clasificar, sino predecir text siguiente.

In [15]:
from transformers import DataCollatorForLanguageModeling  # Define el 'juego' con que estudia el modelo.
from transformers import Trainer, TrainingArguments  # El entrenador oficial de Hugging Face y sus ajustes.


# En Colab hay GPU potente → procesamos 32 chistes de un golpe.
# En una computadora normal → solo 2 (la memoria no da para más).
batch_size = 32 if IN_COLAB else 2

# Cada cuántos pasos mostrar en pantalla cómo va el entrenamiento.
logging_steps = len(tokenized_dataset['train']) // batch_size

# Definimos los parámetros globales de entrenamiento
training_args = TrainingArguments(
    output_dir='./hf-gpt',          # Carpeta donde guarda los modelos parciales.
    num_train_epochs=10,            # Leer el dataset completo 10 veces.
    learning_rate=2e-5,             # Correcciones suaves: es un retoque, no un curso desde cero.
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,              # Freno anti 'aprender de memoria' (regularización).
    eval_strategy='epoch',          # Examen con el 10% al final de cada época.
    save_strategy='epoch',          # Guardar una versión del modelo por cada época.
    load_best_model_at_end=True,    # Conservar al final la versión con mejor nota.
    disable_tqdm=False,             # Mostrar la barrita de progreso.
    logging_steps=logging_steps,
    report_to='none'                # No reportar a servicios externos (wandb, etc.).
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer = Trainer(
    model=model,                    # El modelo a entrenar.
    args=training_args,             # Los ajustes de arriba.
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
                                    # 'mlm=False' → tarea estilo GPT: predecir el siguiente
                                    # token (no tapar palabras al azar como BERT).
    train_dataset=tokenized_dataset['train'],   # El 90% para estudiar.
    eval_dataset=tokenized_dataset['test'],     # El 10% para los exámenes.
)


### 🧠 Explicación: ¡A entrenar!

Una sola línea que dispara todo el proceso: los 10 recorridos por los chistes. Durará varios minutos (en Taller 3, con BERT, eran ~3 min por experimento; aquí algo similar si hay GPU).

La línea `%%time` no es código, es una orden *al notebook*: "cronometra esta celda y muéstrame cuánto tardó al final".

Mientras corre, el modelo va reportando su **loss** (la "pérdida"): qué tan seguido falla al adivinar el siguiente token. **La idea es que baje** con las épocas: es el modelo aprendiendo.

In [16]:
# '%%time' es una orden al notebook (no a Python): cronometrar esta celda
# y mostrar al final cuánto tardó completo.
%%time
# Arranca el entrenamiento: 10 épocas con los chistes.
trainer.train()


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.429856,3.120655
2,2.868066,3.053184
3,2.627920,3.046925
4,2.433456,3.087146
5,2.256299,3.147992
6,2.094557,3.248439
7,1.949440,3.348997
8,1.838623,3.453636
9,1.751734,3.544511
10,1.695912,3.601993


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.99s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


CPU times: user 7min 47s, sys: 8.35 s, total: 7min 55s
Wall time: 8min 18s


TrainOutput(global_step=10890, training_loss=2.2939821893080756, metrics={'train_runtime': 497.9246, 'train_samples_per_second': 43.721, 'train_steps_per_second': 21.871, 'total_flos': 609870912768000.0, 'train_loss': 2.2939821893080756, 'epoch': 10.0})

Ahora observemos los resultados.

### 🧠 Explicación: ¡Resultados! (con el generador oficial)

Ahora el modelo `model` ya no es el original: es la versión con mejor nota después de estudiar los chistes. Le pedimos una generación con el generador oficial de Hugging Face, con `temperature=0.8` (algo creativo) partiendo de la misma frase *"Había una vez"*.

**Compare con la celda 14**: mismos jugadores, pero el modelo estudió chistes — el cambio de estilo debería ser notorio (conversaciones cortas, situaciones puntuales, en vez de prosa literaria).

In [17]:
# Generar texto con el modelo YA entrenado (temperatura 0.8 = moderadamente creativo).
# La frase de entrada 'tokens' sigue siendo la misma de antes: "Había una vez".
output = model.generate(tokens, pad_token_id=tokenizer.eos_token_id, max_length=100, do_sample=True, temperature=0.8)

# Traducir de números a texto y mostrar.
print(tokenizer.decode(output[0]))


Había una vez un ciego con sus gafas. Se paró en la calle y le dice un ciego a otro:
- ¿Qué miras?
- A ti.
- ¿Por qué?
- Pues porque tengo hambre. ¿Y tú?
- ¡Porque no tengo tiempo para comer!
- ¿Por qué? ¿Y eso?
- Porque llevo toda mi vida detrás de un mostrador.
- ¿Y eso qué significa?
- ¡Que me da pena!
-


### 🧠 Explicación: Resultados con NUESTRA función

La misma prueba, pero con la función `generate` que escribimos a mano en la celda 8 — para confirmar que nuestra versión artesanal también funciona sobre el modelo ya entrenado.

`eps=0.2` = "muy codicioso" (solo 20% de las veces sortea): espera un texto bastante conservador y repetitivo, con el acento de chistes que absorbió del fine tuning.

In [18]:
# Misma prueba con nuestra función artesanal, sobre el modelo ya entrenado.
# eps=0.2 → solo el 20% de las veces sortea; casi siempre elige lo más probable.
output_text, _ = generate(model, tokenizer, text, max_length=100, eps=0.2, device=device)

print(output_text)


Había una vez una niña que vivía en un pueblo pequeño. Le preguntó su suegra:
- ¡Abuela,cuánto me alegro de verte!
- Yo también, hija.
- ¡Cuánto tiempo! ¿Cómo estás?
- Dando gracias a Dios, estupenda.
- ¿En qué piensas?
- En que vas a estar mucho por aquí, tan pronto como podamos, ¿no?
- ¡Ajá! ¿Y tú qué ves hijo?
- Pues que vengo a contarte una historia que


No parece ser muy gracioso precisamente, sin embargo, notemos que la generación de texto cambia de "estilo", ahora es mucho más frecuente encontrar conversaciones cortas, frases concisas, y situaciones particulares, en lugar del estilo más literario del modelo original. Esto es un indicio de la influencia que tiene el conjunto de datos de entrenamiento en el modelo final, esto es algo a tener muy en cuenta a la hora de utilizar y hacer fine tuning a modelos de lenguaje.

## Conclusiones
- Los modelos BERT y GPT son muy similares, aunque ambos tienen diferencias claras en cuanto a su estructura y manera de entrenamiento.
- Sin embargo, ambos pueden utilizarse para el mismo tipo de tareas posteriores, lo cual sustenta la importancia del pre-entrenamiento y la construcción de embeddings de buena calidad.
- En los modelos generativos, tiende a ver un dilema de tipo exploración-explitación, al explotar los resultados, podemos ser más precisos, per al mismo tiempo más monótonos, mientras que explorando podemos ser más creativos y diversos, pero al mismo tiempo terminar con texto incoherente, difuso o alucinante. Es necesario evaluar la tarea a la mano para escoger el ajuste adecuado entre estas dos técnicas de decodificación.
- Los modelos generativos de texto no son más que una gran probabilidad de distribución y esta a su vez es completamente dependiente de los datos con los que fue entrenada. Es aquí donde se hace sumamente importante obtener y curar los conjuntos de datos con los que se entrena, de lo contrario se puede terminar con un modelo de mala calidad para la tarea en especifico.
- Diferentes estrategias de decodificación entregan resultados diferentes, vale la pena hacer una exploración de los resultados y ajustar los hiperparámetros para obtener los resultados deseados según el objetivo.